# Mode 02 Kaggle: Z24 NPY dataset - tsai ResNet

Attach the Kaggle Dataset containing `inputs.npy` and `labels.npy`, enable a GPU, then choose **Run All**. The first code cell installs pinned Kaggle-compatible versions without replacing the preinstalled PyTorch/CUDA stack. If an earlier installation attempt changed the environment, restart the Kaggle session before running this notebook.

In [ ]:
# ---------------- Dependencies and imports ----------------
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

def package_version(distribution_name):
    try:
        return version(distribution_name)
    except PackageNotFoundError:
        return None

# tsai 1.x currently asks pip for a newer PyTorch/CUDA stack than Kaggle's
# preinstalled GPU image. Use a compatible tsai release and never replace torch.
PINNED_DEPENDENCIES = {
    'scikit-learn': '1.7.2',
    'imbalanced-learn': '0.14.0',
    'fastai': '2.8.12',
}

installed_tsai = package_version('tsai')
if installed_tsai not in (None, '0.4.1'):
    raise RuntimeError(
        f'This session already contains tsai {installed_tsai} from an incompatible '
        'installation. Choose Session -> Restart Session, then run this notebook '
        'again from the first cell.'
    )

dependency_mismatches = {
    name: (package_version(name), expected)
    for name, expected in PINNED_DEPENDENCIES.items()
    if package_version(name) != expected
}
already_imported = [
    name for name in ('sklearn', 'imblearn', 'fastai', 'tsai')
    if name in sys.modules
]
if dependency_mismatches and already_imported:
    raise RuntimeError(
        f'Packages {already_imported} were already imported with incompatible '
        f'versions {dependency_mismatches}. Choose Session -> Restart Session, '
        'then run this notebook again from the first cell.'
    )

torch_before = package_version('torch')
if dependency_mismatches:
    print('Installing Kaggle-compatible tsai dependencies...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        'scikit-learn==1.7.2',
        'imbalanced-learn==0.14.0',
        'fastai==2.8.12',
        'pyts==0.13.0',
        'psutil>=6.1',
    ])

if installed_tsai is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        '--no-deps', 'tsai==0.4.1',
    ])

importlib.invalidate_caches()
torch_after = package_version('torch')
if torch_after != torch_before:
    raise RuntimeError(
        f'PyTorch changed unexpectedly: {torch_before} -> {torch_after}. '
        'Restart the Kaggle session before continuing.'
    )

from pathlib import Path
from datetime import datetime
import gc
import json
import shutil
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import sklearn
import fastai
import tsai
from IPython.display import display
from fastai.callback.core import Callback
from fastai.metrics import accuracy
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tsai.data.core import TSClassification
from tsai.data.validation import combine_split_data
from tsai.tslearner import TSClassifier
import tsai.inference

print('torch:', torch.__version__)
print('fastai:', fastai.__version__)
print('tsai:', tsai.__version__)
print('scikit-learn:', sklearn.__version__)


In [ ]:
# ---------------- Configuration and device ----------------
MODEL_ARCH = 'ResNet'
BATCH_SIZE = 32
PREDICTION_BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42
TRAIN_SETUPS = 6
VALIDATION_SETUPS = 1
EXPECTED_INPUT_SHAPE = (1530, 27, 6000)
EXPECTED_LABEL_SHAPE = (1530,)
NUM_CLASSES = 17
SETUPS_PER_CONDITION = 9
SEGMENTS_PER_RECORDING = 10
KAGGLE_INPUTS_PATH = Path('/kaggle/input/dataset/inputs.npy')
KAGGLE_LABELS_PATH = Path('/kaggle/input/dataset/labels.npy')

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Selected device:', device)
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for gpu_index in range(torch.cuda.device_count()):
        print(f'GPU {gpu_index}:', torch.cuda.get_device_name(gpu_index))
else:
    print('CUDA is unavailable; training will use CPU.')

KAGGLE_WORKING = Path('/kaggle/working')
WORK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / 'kaggle_working'
CACHE_DIR = WORK_DIR / 'z24_dataset_cache'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_DIR = WORK_DIR / f'z24_tsai_{MODEL_ARCH}_{RUN_ID}'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
print('Output directory:', ARTIFACT_DIR)


In [ ]:
# ---------------- Locate, load, and validate data ----------------
search_roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd(), Path.cwd() / 'raw_data']
search_roots = [path for path in search_roots if path.exists()]
zip_candidates = sorted(
    {candidate.resolve() for root in search_roots for candidate in root.rglob('dataset.zip')},
    key=lambda path: (len(path.parts), str(path)),
)

if KAGGLE_INPUTS_PATH.is_file() and KAGGLE_LABELS_PATH.is_file():
    inputs_path = KAGGLE_INPUTS_PATH
    labels_path = KAGGLE_LABELS_PATH
    print('Using direct Kaggle NPY files:', inputs_path.parent)
elif zip_candidates:
    archive_path = zip_candidates[0]
    inputs_path = CACHE_DIR / 'inputs.npy'
    labels_path = CACHE_DIR / 'labels.npy'
    print('Using archive:', archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        members = {info.filename: info for info in archive.infolist() if not info.is_dir()}
        for member_name, target in (('inputs.npy', inputs_path), ('labels.npy', labels_path)):
            if member_name not in members:
                raise ValueError(f'{archive_path} is missing {member_name}')
            expected_size = members[member_name].file_size
            if target.exists() and target.stat().st_size != expected_size:
                raise ValueError(f'Cached file has the wrong size: {target}')
            if not target.exists():
                temporary = target.with_suffix(target.suffix + '.partial')
                if temporary.exists():
                    temporary.unlink()
                with archive.open(member_name) as source, temporary.open('wb') as destination:
                    shutil.copyfileobj(source, destination, length=16 * 1024 * 1024)
                temporary.replace(target)
else:
    pair = None
    for root in search_roots:
        for candidate in root.rglob('inputs.npy'):
            sibling = candidate.with_name('labels.npy')
            if sibling.exists() and CACHE_DIR not in candidate.parents:
                pair = (candidate, sibling)
                break
        if pair:
            break
    if pair is None:
        raise FileNotFoundError(
            'Could not find /kaggle/input/dataset/inputs.npy and labels.npy, '
            'or a dataset.zip archive.'
        )
    inputs_path, labels_path = pair
    print('Using discovered NPY files:', inputs_path.parent)

inputs = np.load(inputs_path, mmap_mode='r', allow_pickle=False)
labels = np.load(labels_path, allow_pickle=False)
if inputs.shape != EXPECTED_INPUT_SHAPE or inputs.dtype != np.float32:
    raise ValueError(f'Expected float32 inputs {EXPECTED_INPUT_SHAPE}, got {inputs.shape} {inputs.dtype}')
if labels.shape != EXPECTED_LABEL_SHAPE or labels.dtype != np.int64:
    raise ValueError(f'Expected int64 labels {EXPECTED_LABEL_SHAPE}, got {labels.shape} {labels.dtype}')
expected_labels = np.repeat(
    np.arange(NUM_CLASSES, dtype=np.int64),
    SETUPS_PER_CONDITION * SEGMENTS_PER_RECORDING,
)
if not np.array_equal(labels, expected_labels):
    raise ValueError('Labels are not ordered as 17 conditions x 9 setups x 10 segments')
print('Dataset:', inputs.shape, inputs.dtype, '| labels:', labels.shape, labels.dtype)


In [ ]:
# ---------------- Leakage-safe setup-grouped split ----------------
sample_index = np.arange(len(labels), dtype=np.int64)
within_condition = sample_index % (SETUPS_PER_CONDITION * SEGMENTS_PER_RECORDING)
setup_id = within_condition // SEGMENTS_PER_RECORDING
segment_id = within_condition % SEGMENTS_PER_RECORDING
recording_id = labels * SETUPS_PER_CONDITION + setup_id

setup_order = np.arange(SETUPS_PER_CONDITION, dtype=np.int64)
np.random.default_rng(SEED).shuffle(setup_order)
setup_split = {
    'train': setup_order[:TRAIN_SETUPS],
    'validation': setup_order[TRAIN_SETUPS:TRAIN_SETUPS + VALIDATION_SETUPS],
    'test': setup_order[TRAIN_SETUPS + VALIDATION_SETUPS:],
}
split_indexes = {
    name: np.flatnonzero(np.isin(setup_id, selected_setups))
    for name, selected_setups in setup_split.items()
}
recording_sets = {
    name: set(recording_id[indexes].tolist())
    for name, indexes in split_indexes.items()
}
assert recording_sets['train'].isdisjoint(recording_sets['validation'])
assert recording_sets['train'].isdisjoint(recording_sets['test'])
assert recording_sets['validation'].isdisjoint(recording_sets['test'])

split_summary = []
for name, indexes in split_indexes.items():
    split_summary.append({
        'split': name,
        'segments': len(indexes),
        'recordings': len(recording_sets[name]),
        'setups': setup_split[name].tolist(),
        'conditions': len(np.unique(labels[indexes])),
    })
display(pd.DataFrame(split_summary).set_index('split'))


In [ ]:
# ---------------- Materialize tsai arrays and normalize ----------------
def materialize_tsai(indexes, chunk_size=64):
    result = np.empty((len(indexes), inputs.shape[1], inputs.shape[2]), dtype=np.float32)
    for start in range(0, len(indexes), chunk_size):
        selected = indexes[start:start + chunk_size]
        result[start:start + len(selected)] = np.asarray(inputs[selected], dtype=np.float32)
    return result, np.asarray(labels[indexes], dtype=np.int64)

X_train, y_train = materialize_tsai(split_indexes['train'])
X_validation, y_validation = materialize_tsai(split_indexes['validation'])
X_test, y_test = materialize_tsai(split_indexes['test'])

# Compute per-sensor statistics from train only; never fit preprocessing on val/test.
sensor_mean = X_train.mean(axis=(0, 2), keepdims=True, dtype=np.float64).astype(np.float32)
sensor_std = X_train.std(axis=(0, 2), keepdims=True, dtype=np.float64).astype(np.float32)
sensor_std = np.where(sensor_std < 1e-8, 1.0, sensor_std).astype(np.float32)
for array in (X_train, X_validation, X_test):
    array -= sensor_mean
    array /= sensor_std

for name, X_part, y_part in (
    ('train', X_train, y_train),
    ('validation', X_validation, y_validation),
    ('test', X_test, y_test),
):
    assert X_part.shape[1:] == (27, 6000)
    assert X_part.dtype == np.float32
    assert np.isfinite(X_part).all()
    assert set(np.unique(y_part)) == set(range(NUM_CLASSES))
    print(f'{name:10s}: X={X_part.shape}, y={y_part.shape}, RAM={X_part.nbytes / 1024**2:.1f} MiB')

del inputs
gc.collect()


In [ ]:
# ---------------- Build tsai learner ----------------
X, y, tsai_splits = combine_split_data(
    [X_train, X_validation], [y_train, y_validation]
)
learner = TSClassifier(
    X,
    y,
    splits=tsai_splits,
    tfms=[None, TSClassification()],
    arch=MODEL_ARCH,
    metrics=accuracy,
    bs=BATCH_SIZE,
    wd=WEIGHT_DECAY,
    path=ARTIFACT_DIR,
    model_dir='models',
    seed=SEED,
    device=device,
    verbose=True,
)
learner.summary()


In [ ]:
# ---------------- Training and timing ----------------
def synchronize_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


class EpochTimerCallback(Callback):
    order = 100

    def before_fit(self):
        self.epoch_seconds = []

    def before_epoch(self):
        synchronize_device()
        self.epoch_started = time.perf_counter()

    def after_epoch(self):
        synchronize_device()
        self.epoch_seconds.append(time.perf_counter() - self.epoch_started)


epoch_timer = EpochTimerCallback()
synchronize_device()
training_started = time.perf_counter()
learner.fit_one_cycle(EPOCHS, LEARNING_RATE, cbs=[epoch_timer])
synchronize_device()
training_seconds_total = time.perf_counter() - training_started

epoch_seconds = np.asarray(epoch_timer.epoch_seconds, dtype=np.float64)
if len(epoch_seconds) == 0:
    raise RuntimeError('Epoch timing callback did not record any epochs')
steady_epoch_seconds = epoch_seconds[1:] if len(epoch_seconds) > 1 else epoch_seconds
mean_epoch_seconds = float(epoch_seconds.mean())
mean_epoch_seconds_excluding_first = float(steady_epoch_seconds.mean())
effective_train_samples_per_second = float(
    len(X_train) / mean_epoch_seconds_excluding_first
)
print(f'Total training time: {training_seconds_total:.2f} seconds')
print(f'Mean epoch time: {mean_epoch_seconds:.2f} seconds')
print(
    'Mean epoch time excluding first: '
    f'{mean_epoch_seconds_excluding_first:.2f} seconds'
)
print(
    'Effective train throughput: '
    f'{effective_train_samples_per_second:.2f} samples/second '
    '(epoch time includes validation)'
)


In [ ]:
# ---------------- Train / validation / test metrics ----------------
def evaluate_split(split_name, X_part, y_part):
    probabilities, _, _ = learner.get_X_preds(
        X_part, y_part, bs=PREDICTION_BATCH_SIZE, with_decoded=True
    )
    if hasattr(probabilities, 'detach'):
        probabilities = probabilities.detach().cpu().numpy()
    else:
        probabilities = np.asarray(probabilities)
    predictions = probabilities.argmax(axis=1).astype(np.int64)
    targets = np.asarray(y_part, dtype=np.int64).reshape(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        targets, predictions, average='macro', zero_division=0
    )
    return {
        'split': split_name,
        'accuracy': accuracy_score(targets, predictions),
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1,
    }, predictions

metric_rows = []
predictions_by_split = {}
for split_name, X_part, y_part in (
    ('train', X_train, y_train),
    ('validation', X_validation, y_validation),
    ('test', X_test, y_test),
):
    row, split_predictions = evaluate_split(split_name, X_part, y_part)
    metric_rows.append(row)
    predictions_by_split[split_name] = split_predictions

split_metrics = pd.DataFrame(metric_rows).set_index('split')
print('Final metrics - precision/recall/F1 are macro averages:')
display(split_metrics.style.format('{:.2%}'))

test_predictions = predictions_by_split['test']
test_report = classification_report(
    y_test, test_predictions, labels=np.arange(NUM_CLASSES),
    output_dict=True, zero_division=0,
)
test_matrix = confusion_matrix(y_test, test_predictions, labels=np.arange(NUM_CLASSES))

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
optimizer_name = getattr(learner.opt_func, '__name__', learner.opt_func.__class__.__name__)
model_dtype = str(next(learner.model.parameters()).dtype).replace('torch.', '')
benchmark_summary = {
    'pipeline': 'tsai_resnet',
    'training_seconds_total': float(training_seconds_total),
    'epochs_completed': int(len(epoch_seconds)),
    'mean_epoch_seconds': float(mean_epoch_seconds),
    'mean_epoch_seconds_excluding_first': float(mean_epoch_seconds_excluding_first),
    'effective_train_samples_per_second': float(effective_train_samples_per_second),
    'model_parameters': int(sum(parameter.numel() for parameter in learner.model.parameters())),
    'trainable_parameters': int(sum(
        parameter.numel() for parameter in learner.model.parameters()
        if parameter.requires_grad
    )),
    'batch_size': int(BATCH_SIZE),
    'replicas': 1,
    'gpu_names': ' | '.join(gpu_names) if gpu_names else 'CPU',
    'pytorch_version': torch.__version__,
    'fastai_version': fastai.__version__,
    'tsai_version': tsai.__version__,
    'precision_policy': model_dtype,
    'model_architecture': MODEL_ARCH,
    'model_input_shape': '27x6000',
    'input_layout': 'sensors_x_time_steps',
    'normalization': 'train_sensor_zscore',
    'optimizer': optimizer_name,
    'data_pipeline': 'tsai_fastai_dataloaders',
    'early_stopping_monitor': 'none_fixed_epochs',
    'regularization': f'weight_decay_{WEIGHT_DECAY}',
    'split_strategy': 'setup_grouped_6_1_2',
}
for split_name in ('train', 'validation', 'test'):
    for metric_name in ('accuracy', 'precision_macro', 'recall_macro', 'f1_macro'):
        benchmark_summary[f'{split_name}_{metric_name}'] = float(
            split_metrics.loc[split_name, metric_name]
        )

benchmark_frame = pd.DataFrame([benchmark_summary]).set_index('pipeline')
print()
print('Standardized benchmark summary:')
display(benchmark_frame.T)


In [ ]:
# ---------------- Save model, reports, plots, and ZIP ----------------
learner.export('z24_dataset_zip_tsai.pkl')
split_metrics.to_csv(ARTIFACT_DIR / 'split_metrics.csv')
benchmark_frame.to_csv(ARTIFACT_DIR / 'benchmark_summary.csv')
(ARTIFACT_DIR / 'benchmark_summary.json').write_text(
    json.dumps(benchmark_summary, indent=2), encoding='utf-8'
)
pd.DataFrame({
    'epoch': np.arange(1, len(epoch_seconds) + 1),
    'seconds': epoch_seconds,
}).to_csv(ARTIFACT_DIR / 'epoch_times.csv', index=False)
pd.DataFrame(test_report).T.to_csv(ARTIFACT_DIR / 'test_classification_report.csv')
np.savetxt(ARTIFACT_DIR / 'test_confusion_matrix.csv', test_matrix, fmt='%d', delimiter=',')
np.savez(
    ARTIFACT_DIR / 'preprocessing_and_splits.npz',
    sensor_mean=sensor_mean.reshape(-1),
    sensor_std=sensor_std.reshape(-1),
    train_indexes=split_indexes['train'],
    validation_indexes=split_indexes['validation'],
    test_indexes=split_indexes['test'],
)

recorder_values = learner.recorder.values
history_columns = list(learner.recorder.metric_names[1:-1])
if recorder_values:
    if len(history_columns) != len(recorder_values[0]):
        history_columns = [f'metric_{index}' for index in range(len(recorder_values[0]))]
    history_frame = pd.DataFrame(recorder_values, columns=history_columns)
    history_frame.insert(0, 'epoch', np.arange(1, len(history_frame) + 1))
else:
    history_frame = pd.DataFrame()
history_frame.to_csv(ARTIFACT_DIR / 'history.csv', index=False)

experiment = {
    'pipeline': 'tsai_resnet',
    'data_source': str(inputs_path),
    'model_architecture': MODEL_ARCH,
    'data_contract': 'samples x sensors x time_samples',
    'train_shape': list(X_train.shape),
    'validation_shape': list(X_validation.shape),
    'test_shape': list(X_test.shape),
    'num_classes': NUM_CLASSES,
    'seed': SEED,
    'epochs_requested': EPOCHS,
    'epochs_completed': int(len(epoch_seconds)),
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'setup_split': {name: values.tolist() for name, values in setup_split.items()},
    'device': str(device),
    'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'pytorch_version': torch.__version__,
    'normalization': 'train_sensor_zscore',
    'split_strategy': 'setup_grouped_6_1_2',
    'metrics': split_metrics.to_dict(orient='index'),
    'benchmark': benchmark_summary,
}
(ARTIFACT_DIR / 'experiment.json').write_text(
    json.dumps(experiment, indent=2), encoding='utf-8'
)

if not history_frame.empty:
    plot_columns = [name for name in ('train_loss', 'valid_loss', 'accuracy') if name in history_frame]
    if plot_columns:
        ax = history_frame.plot(x='epoch', y=plot_columns, figsize=(10, 5), grid=True)
        ax.set_title('tsai training history')
        ax.figure.tight_layout()
        ax.figure.savefig(ARTIFACT_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
        plt.show()

fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(test_matrix, display_labels=np.arange(NUM_CLASSES)).plot(
    ax=ax, cmap='Blues', colorbar=False
)
ax.set_title('Mode 02 tsai test confusion matrix')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'test_confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()

archive_output = shutil.make_archive(str(ARTIFACT_DIR), 'zip', root_dir=ARTIFACT_DIR)
print('Completed successfully.')
print('Artifacts:', ARTIFACT_DIR)
print('Download ZIP:', archive_output)
